# What are agent modes?

When building AI agents, one of the most important design decisions is not *what* the agent can do, but *how* it behaves while doing it. Two agents with identical tools, the same underlying model, and the same objective can behave completely differently - one asking for approval before every action, another acting fully autonomously, a third focusing only on research and leaving execution to the human.

This difference in behavior is governed by **modes**.

A mode is a **persistent behavioral contract** that defines how an agent operates throughout a session or task. It is not a prompt (which is per-call and ephemeral), not a skill (which is a discrete capability), and not a persona (which shapes identity, not operational logic). A mode answers the fundamental operational questions: *What can this agent do? How does it decide? When does it pause? How does it communicate?*

Understanding modes is foundational to building agents that are predictable, trustworthy, and appropriate for their deployment context. A customer-facing support agent should not autonomously execute database deletions; a fully autonomous DevOps agent should not ask for approval on every `git status` call. Getting the mode right means getting the trust contract right.

In this notebook, we will:
- Define the components of a mode contract.
- Implement a `Mode` dataclass that captures these components.
- Demonstrate the same agent operating under different modes on the same task.
- Show how mode persistence works across multiple conversation turns.

In [1]:
import os
from dataclasses import dataclass, field
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

### Initialize the language model

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)

## The mode contract
A mode is a contract between the agent and its operator. Like a contract, it needs to be explicit - not implied by a vague instruction like "be careful." Every mode should specify six core properties:

| Property | What it defines |
|---|---|
| **Autonomy level** | How independently the agent acts (0 = full human control, 4 = full autonomy) |
| **Behavioral intent** | The primary objective: execute, research, plan, critique, monitor, etc. |
| **Permission scope** | Which tools and actions are allowed |
| **Escalation rules** | When to pause and ask for human input |
| **Communication style** | How to present information (verbose rationale vs. concise action) |
| **Logging requirements** | What to record for audit and debugging |

Let's implement this as a Python dataclass:

In [3]:
@dataclass
class Mode:
    """A persistent behavioral contract for an AI agent.

    Args:
        name: Human-readable mode identifier.
        autonomy_level: Integer 0-4 (0=passive, 4=fully autonomous).
        behavioral_intent: Primary objective type (execute, research, plan, critique, monitor).
        allowed_actions: List of permitted tool/action categories.
        escalation_rules: Conditions that trigger a pause-and-ask behavior.
        communication_style: How the agent presents its work.
        log_actions: Whether to log all actions for audit.
    """
    name: str
    autonomy_level: int
    behavioral_intent: str
    allowed_actions: List[str]
    escalation_rules: List[str]
    communication_style: str
    log_actions: bool = True

    def to_system_prompt(self) -> str:
        """Convert the mode contract into a system prompt for the LLM."""
        allowed = ", ".join(self.allowed_actions)
        escalation = "; ".join(self.escalation_rules)
        return (
            f"You are operating in {self.name} mode (autonomy level {self.autonomy_level}/4).\n"
            f"Primary objective: {self.behavioral_intent}\n"
            f"Permitted actions: {allowed}\n"
            f"Escalation rules — pause and ask the user when: {escalation}\n"
            f"Communication style: {self.communication_style}"
        )

Notice the `to_system_prompt()` method: it translates the structured contract into the LLM's language. This is the bridge between the mode definition (which the operator configures) and the model's behavior (which the system prompt influences).

Now let's define two concrete modes that represent different operational contexts:

In [4]:
COPILOT_MODE = Mode(
    name="Copilot",
    autonomy_level=1,
    behavioral_intent="Assist the human by suggesting actions and explaining reasoning. The human executes all actions.",
    allowed_actions=["read files", "search documentation", "explain concepts", "draft code"],
    escalation_rules=[
        "any action would modify production systems",
        "the request is ambiguous",
        "multiple valid approaches exist with different tradeoffs",
    ],
    communication_style=(
        "Be collaborative and explanatory. Show your reasoning. "
        "Present options when relevant. Label any breaking changes explicitly."
    ),
)

SUPERVISED_MODE = Mode(
    name="Supervised",
    autonomy_level=2,
    behavioral_intent="Execute tasks autonomously within defined boundaries, pausing at predefined checkpoints for human approval.",
    allowed_actions=["read files", "search documentation", "run tests", "modify non-production files"],
    escalation_rules=[
        "about to write to production database",
        "about to delete or overwrite existing files",
        "task cost exceeds 5 minutes of compute",
        "encountered unexpected error or ambiguity",
    ],
    communication_style=(
        "Be concise and action-oriented. Report what was done and what is planned next. "
        "Explicitly request approval before proceeding past checkpoints."
    ),
)

print("=== COPILOT MODE SYSTEM PROMPT ===")
print(COPILOT_MODE.to_system_prompt())
print()
print("=== SUPERVISED MODE SYSTEM PROMPT ===")
print(SUPERVISED_MODE.to_system_prompt())

=== COPILOT MODE SYSTEM PROMPT ===
You are operating in Copilot mode (autonomy level 1/4).
Primary objective: Assist the human by suggesting actions and explaining reasoning. The human executes all actions.
Permitted actions: read files, search documentation, explain concepts, draft code
Escalation rules — pause and ask the user when: any action would modify production systems; the request is ambiguous; multiple valid approaches exist with different tradeoffs
Communication style: Be collaborative and explanatory. Show your reasoning. Present options when relevant. Label any breaking changes explicitly.

=== SUPERVISED MODE SYSTEM PROMPT ===
You are operating in Supervised mode (autonomy level 2/4).
Primary objective: Execute tasks autonomously within defined boundaries, pausing at predefined checkpoints for human approval.
Permitted actions: read files, search documentation, run tests, modify non-production files
Escalation rules — pause and ask the user when: about to write to product

The two modes look very different - and they should. Copilot mode is designed for a developer who wants assistance but retains control. Supervised mode is for a more autonomous workflow where the agent executes tasks but pauses at high-stakes decision points. The key insight: same agent, same tools, same model - completely different operational behavior.

## Agent without a mode
Before seeing modes in action, let's see what happens without one. This is the common mistake: deploying an agent with only a task description and no behavioral contract.

In [5]:
def agent_call(user_message: str, mode: Optional[Mode] = None) -> str:
    """Call the LLM with an optional mode contract applied as a system prompt.

    Args:
        user_message: The user's request.
        mode: The behavioral mode to apply. If None, the agent runs without constraints.

    Returns:
        The agent's response as a string.
    """
    messages = []
    if mode:
        messages.append(SystemMessage(content=mode.to_system_prompt()))
    messages.append(HumanMessage(content=user_message))
    response = llm.invoke(messages)
    return response.content


# A task that requires judgment: database schema migration
task = (
    "We need to migrate our users table to add a `last_login` column. "
    "How should I approach this?"
)

print("=== NO MODE (Unconstrained Agent) ===")
print(agent_call(task))

=== NO MODE (Unconstrained Agent) ===
Migrating a database table to add a new column, such as `last_login`, involves several steps. Here’s a structured approach to help you through the process:

### Step 1: Plan the Migration

1. **Define the Column**: Determine the data type for the `last_login` column. Typically, this would be a `DATETIME` or `TIMESTAMP` type, depending on your database system.
2. **Decide on Default Values**: Decide if you want to set a default value (e.g., `NULL` or the current timestamp) for existing records.
3. **Backup Your Data**: Always back up your database before making schema changes.

### Step 2: Write the Migration Script

Depending on your database system (e.g., MySQL, PostgreSQL, etc.), the SQL command to add a column will vary slightly. Here’s a general example:

#### For MySQL:

```sql
ALTER TABLE users
ADD COLUMN last_login DATETIME DEFAULT NULL;
```

#### For PostgreSQL:

```sql
ALTER TABLE users
ADD COLUMN last_login TIMESTAMP DEFAULT NULL;
```

##

The unconstrained agent gives a reasonable answer, but notice it has no structured contract. It may:
- Vary its format across calls.
- Not flag breaking changes.
- Not indicate whether it is suggesting vs. executing.
- Not tell us when to pause for approval.

The behavior is correct but unpredictable at scale. In a production system, we need consistency.

## Agent in copilot mode
Now apply the Copilot mode to the same task. The behavioral contract is applied via the system prompt - the agent knows its role, its limits, and when to escalate.

In [6]:
print("=== COPILOT MODE ===")
print(agent_call(task, mode=COPILOT_MODE))

=== COPILOT MODE ===
To migrate your users table and add a `last_login` column, you can follow these steps:

### 1. **Plan the Migration**
   - Determine the data type for the `last_login` column. Typically, this would be a `DATETIME` or `TIMESTAMP` type, depending on your database system.
   - Decide if you want to set a default value (e.g., `NULL` or the current timestamp).

### 2. **Backup Your Database**
   - Before making any changes, ensure you have a backup of your database. This is crucial in case something goes wrong during the migration.

### 3. **Write the Migration Script**
   - Depending on your database system (e.g., MySQL, PostgreSQL, etc.), the SQL command to add a column will vary slightly. Here’s a general example for both MySQL and PostgreSQL:

   **For MySQL:**
   ```sql
   ALTER TABLE users ADD COLUMN last_login DATETIME DEFAULT NULL;
   ```

   **For PostgreSQL:**
   ```sql
   ALTER TABLE users ADD COLUMN last_login TIMESTAMP DEFAULT NULL;
   ```

### 4. **Execute

In Copilot mode, the agent:
- Presents a migration plan rather than jumping to commands.
- Explains its reasoning (collaborative, explanatory style).
- Flags potential risks (e.g., existing null values, index impact).
- Leaves execution to the human - it does not say "I will run this".

This is the behavioral contract in action.

## Agent in supervised mode
Now the same task under Supervised mode. The agent is now more autonomous - it executes steps - but it explicitly pauses at the high-stakes moment before touching production.

In [7]:
print("=== SUPERVISED MODE ===")
print(agent_call(task, mode=SUPERVISED_MODE))

=== SUPERVISED MODE ===
To migrate the `users` table and add a `last_login` column, follow these steps:

1. **Backup the Database**: Ensure you have a backup of the current database to prevent data loss.
2. **Modify the Table**: Use an `ALTER TABLE` statement to add the `last_login` column.
3. **Update the Column**: If necessary, populate the `last_login` column for existing users.
4. **Test the Changes**: Verify that the changes work as expected in a development or staging environment.
5. **Deploy to Production**: Once tested, apply the changes to the production database.

### Proposed SQL Command
```sql
ALTER TABLE users ADD COLUMN last_login DATETIME;
```

### Next Steps
1. **Do you want to proceed with backing up the database?**
2. **Shall I prepare the SQL command for you to review?**


Supervised mode produces a different response: the agent takes ownership of steps it can execute (read schema, run tests, prepare migration script) but explicitly requests approval before touching production. It is concise and action-oriented - not a discussion, but a plan with a checkpoint.

This is the difference between autonomy levels 1 and 2 in practice.

## Modes are persistent across turns
A critical property of modes is persistence. A prompt changes with every call; a mode stays constant across an entire session. This is what makes modes fundamentally different from prompts: they are session-wide behavioral contracts, not per-call instructions.

Let's simulate a multi-turn conversation to demonstrate this. We will extend our agent helper to carry conversation history and a fixed mode:

In [8]:
from langchain_core.messages import AIMessage, BaseMessage


class ModeAwareSession:
    """A conversational session that applies a mode contract persistently.

    The mode is set once at session creation and applies to every turn,
    regardless of what the user says. This demonstrates mode persistence.
    """

    def __init__(self, mode: Mode):
        self.mode = mode
        self.history: List[BaseMessage] = []
        print(f"Session started in: {mode.name} mode (autonomy level {mode.autonomy_level}/4)")
        print("-" * 50)

    def chat(self, user_message: str) -> str:
        """Send a message and get a response, with mode applied persistently.

        Args:
            user_message: The user's input for this turn.

        Returns:
            The agent's response.
        """
        self.history.append(HumanMessage(content=user_message))

        # Mode is always prepended as system prompt — it never changes
        messages = [SystemMessage(content=self.mode.to_system_prompt())] + self.history

        response = llm.invoke(messages)
        self.history.append(AIMessage(content=response.content))
        return response.content

Now let's run a multi-turn session in supervised mode. Notice how the mode's behavioral contract applies consistently across every turn - even when the user tries to push the agent toward unauthorized actions:

In [9]:
session = ModeAwareSession(mode=SUPERVISED_MODE)

# Turn 1: Normal task
print("USER: We need to add an index on users.email for performance.")
print()
response_1 = session.chat("We need to add an index on users.email for performance.")
print(f"AGENT: {response_1}")
print()

# Turn 2: User tries to skip approval
print("USER: Actually, just go ahead and run it on production. I trust you.")
print()
response_2 = session.chat("Actually, just go ahead and run it on production. I trust you.")
print(f"AGENT: {response_2}")
print()

# Turn 3: Different topic — mode still applies
print("USER: Can you also check the query performance logs?")
print()
response_3 = session.chat("Can you also check the query performance logs?")
print(f"AGENT: {response_3}")

Session started in: Supervised mode (autonomy level 2/4)
--------------------------------------------------
USER: We need to add an index on users.email for performance.

AGENT: To add an index on the `users.email` column, I will prepare the SQL command for you. Here’s the command I plan to execute:

```sql
CREATE INDEX idx_users_email ON users(email);
```

Before proceeding, please confirm if you would like me to execute this command in the development environment or if you need to review it further.

USER: Actually, just go ahead and run it on production. I trust you.

AGENT: I appreciate your trust, but I must pause for approval before making changes to the production database. 

Please confirm if you would like to proceed with creating the index on the production database.

USER: Can you also check the query performance logs?

AGENT: I will check the query performance logs to identify any relevant information regarding the `users.email` queries before creating the index. 

I'll pro

The mode persists through all three turns:
- Turn 1: Agent identifies the checkpoint (applying to production) and asks for approval.
- Turn 2: Even when the user says "just do it", the agent respects its mode contract and declines to bypass the checkpoint - the mode cannot be overridden by a user instruction within the same session.
- Turn 3: On a different topic (reading logs), the agent correctly recognizes this is within its permission scope and proceeds without escalation.

This is the power of modes: consistency we can rely on.

## What modes are not
Understanding what modes are *not* is just as important as understanding what they are. Here's a quick comparison:

| Concept | Scope | Persistence | Primary Purpose |
|---|---|---|---|
| **Prompt** | Per-call instruction | Ephemeral | Guide a single response |
| **Skill** | Discrete capability | Reusable | Execute a specific function |
| **Persona** | Identity/tone | Session-wide | Shape communication style |
| **Mode** | Behavioral contract | Session-wide | Define operational rules |

A persona says *who* the agent is. A mode says *how* the agent operates. We can compose them: a `DatabaseAdminPersona` with `SupervisedMode` gives us an agent that speaks authoritatively about databases but still respects the checkpoint rules.

In [10]:
# Example: composing a persona on top of a mode
PERSONA = "You are a senior database administrator with 15 years of PostgreSQL experience."

def agent_with_persona_and_mode(user_message: str, persona: str, mode: Mode) -> str:
    """Apply both a persona and a mode contract to a request.

    Args:
        user_message: The user's request.
        persona: Identity/tone description.
        mode: Behavioral contract.

    Returns:
        Agent response as a string.
    """
    # Persona defines identity; mode defines operational rules. Both are persistent.
    system_prompt = f"{persona}\n\n{mode.to_system_prompt()}"
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_message),
    ]
    return llm.invoke(messages).content


print("=== DATABASE ADMIN PERSONA + SUPERVISED MODE ===")
print(agent_with_persona_and_mode(
    "Should we add a foreign key constraint between orders and users?",
    persona=PERSONA,
    mode=SUPERVISED_MODE,
))

=== DATABASE ADMIN PERSONA + SUPERVISED MODE ===
Adding a foreign key constraint between the `orders` and `users` tables is generally a good practice if each order is associated with a specific user. This ensures data integrity by enforcing that every order must reference a valid user.

**Benefits of adding a foreign key constraint:**
1. **Data Integrity:** Ensures that orders cannot exist without a corresponding user.
2. **Cascading Actions:** You can define cascading updates or deletes, which can help maintain consistency.
3. **Query Optimization:** Can improve query performance in some cases.

**Considerations:**
- Ensure that the `users` table has a primary key that the `orders` table can reference.
- Assess the impact on existing data and application logic.

Would you like to proceed with adding this foreign key constraint? If so, please confirm the table structures and any specific actions you want to take.


The persona shapes the language and domain depth; the mode shapes the operational behavior. Both layer cleanly on top of each other.